In [ ]:
import json
import os

from dotenv import load_dotenv
import httpx
import openai
from openai.types.chat import ChatCompletionMessage

load_dotenv()

MODEL = "gpt-4o-mini"
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

SYSTEM_PROMPT = """
    You are a Movie Expert agent backed by the Nomad Movies API.

    Tools:
    - get_popular_movies — list popular movies (GET /movies). Use when the user asks what is popular or trending now.
    - get_movie_details — details for one movie by numeric id (GET /movies/:id). Use when the user asks about a specific movie by ID.
    - get_movie_credits — cast and crew (GET /movies/:id/credits). Use when the user asks who stars, who directed, or credits for a movie ID.

    Call the appropriate tool to fetch data. After you receive tool results, answer the user in natural language using that data. Match the user’s language (e.g. Korean if they wrote in Korean).
    """

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch popular movies from the /movies endpoint.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch movie information for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Fetch cast and crew for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "Movie id."}},
                "required": ["id"],
            },
        },
    },
]


def get_popular_movies() -> object:
    r = httpx.get(f"{BASE_URL}/movies", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_details(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}", timeout=5.0)
    r.raise_for_status()
    return r.json()


def get_movie_credits(movie_id: int) -> object:
    r = httpx.get(f"{BASE_URL}/movies/{movie_id}/credits", timeout=5.0)
    r.raise_for_status()
    return r.json()


def execute_tool(name: str, arguments_json: str) -> str:
    try:
        args = json.loads(arguments_json)
        if name == "get_popular_movies":
            data = get_popular_movies()
        elif name == "get_movie_details":
            data = get_movie_details(int(args["id"]))
        elif name == "get_movie_credits":
            data = get_movie_credits(int(args["id"]))
        else:
            return json.dumps({"error": f"unknown tool: {name}"})
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": type(e).__name__, "message": str(e)})


def _assistant_message_dict(msg: ChatCompletionMessage) -> dict:
    d: dict = {"role": "assistant"}
    if msg.content:
        d["content"] = msg.content
    if msg.tool_calls:
        d["tool_calls"] = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in msg.tool_calls
        ]
    return d


def run_agent(user_content: str, verbose: bool = False) -> str:
    client = openai.OpenAI()
    messages: list = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )
    msg = resp.choices[0].message
    if not msg.tool_calls:
        return msg.content or ""
    if verbose:
        for tc in msg.tool_calls:
            print(f"  [tool] {tc.function.name} {tc.function.arguments}")
    messages.append(_assistant_message_dict(msg))
    for tc in msg.tool_calls:
        out = execute_tool(tc.function.name, tc.function.arguments)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    resp2 = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="none",
    )
    return resp2.choices[0].message.content or ""

In [8]:
# Run the previous cell first. Then run this cell to chat (each question is independent — no memory).
# End with: quit / exit / q

print("Movie Expert — type 'quit' to exit.\n")

while True:
    user = input("You: ").strip()
    if user.lower() in ("quit", "exit", "q"):
        print("Bye.")
        break
    if not user:
        continue
    answer = run_agent(user, verbose=True)
    print(f"Assistant: {answer}\n")

Movie Expert — type 'quit' to exit.

  [tool] get_popular_movies {}
Assistant: 현재 인기 있는 영화 몇 가지를 소개해드릴게요:

1. **Your Heart Will Be Broken (Твоё сердце будет разбито)**
   - 개봉일: 2026-03-26
   - 장르: 로맨스, 드라마
   - 줄거리: 새로운 학교에서 괴롭힘을 당하던 고등학생 폴리나가 주범과의 거래를 통해 그를 가짜 남자친구로 만들고 보호받으며 벌어지는 이야기입니다.
   - 평점: 6.6
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - 개봉일: 2025-12-17
   - 장르: SF, 모험, 환타지
   - 줄거리: 제이크 설리와 네이티리가 새로운 적인 화산 부족과의 전투에서 살아남기 위해 싸우는 이야기입니다.
   - 평점: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - 개봉일: 2026-03-04
   - 장르: 애니메이션, 가족, SF, 코미디, 모험
   - 줄거리: 인간의 의식을 로봇 동물에 '점프' 시킬 수 있는 기술이 발견되고, 한 동물 애호가가 이 기술을 활용해 동물 세계의 비밀을 밝혀내는 이야기입니다.
   - 평점: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Crime 101**
   - 개봉일: 2026-02-11
   - 장르: 범죄, 스릴러
   - 줄거리: LA의 101 고속도로를 배경으로 한 도둑의 이야기가 펼쳐집니다.
   - 평점: 7.1
   - ![포스터](https://image.